# 🔍 DeepSight — Training on Google Colab T4 GPU

This notebook trains the DeepSight dual-branch image forgery detection model.

### Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload the **CASIA_v2** dataset to Google Drive (see Cell 3)
3. Upload or clone this project to Colab (see Cell 2)

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Mount Google Drive & set project path ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️  Set this to where you uploaded the deepsight project folder in Drive:
#     e.g. if it's at MyDrive/deepsight, leave as below.
PROJECT_DIR = '/content/drive/MyDrive/deepsight'

# Alternative: clone from GitHub (uncomment and set your repo URL)
# !git clone https://github.com/YOUR_USERNAME/deepsight.git /content/deepsight
# PROJECT_DIR = '/content/deepsight'

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Project files:', os.listdir('.'))

In [ ]:
# ── Cell 3: Dataset setup ─────────────────────────────────────────────────────
# Option A (Recommended): Link your CASIA_v2 dataset from Drive
# Upload CASIA_v2 to Drive at: MyDrive/datasets/CASIA_v2/
# It should have: CASIA_v2/Au_jpg/  and  CASIA_v2/Tp_jpg/

import os

DATASET_DRIVE_PATH = '/content/drive/MyDrive/datasets/CASIA_v2'
DATASET_LOCAL_PATH = 'data/raw/CASIA_v2'

os.makedirs('data/raw', exist_ok=True)

if os.path.exists(DATASET_DRIVE_PATH):
    # Create a symlink so the project code finds the data at the expected path
    if not os.path.exists(DATASET_LOCAL_PATH):
        os.symlink(DATASET_DRIVE_PATH, DATASET_LOCAL_PATH)
        print(f'Symlinked dataset: {DATASET_DRIVE_PATH} → {DATASET_LOCAL_PATH}')
    else:
        print(f'Dataset already linked at {DATASET_LOCAL_PATH}')
    print('Au_jpg files:', len(os.listdir(f'{DATASET_LOCAL_PATH}/Au_jpg')))
    print('Tp_jpg files:', len(os.listdir(f'{DATASET_LOCAL_PATH}/Tp_jpg')))
else:
    print(f'ERROR: Dataset not found at {DATASET_DRIVE_PATH}')
    print('Please upload CASIA_v2 to your Google Drive at that path.')

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────────
# Colab already has torch/torchvision with CUDA — only install the extras.
!pip install -q \
    timm>=0.9.0 \
    albumentations>=2.0.0 \
    pillow-heif>=0.13.0 \
    scikit-image>=0.22.0

print('Dependencies installed.')

In [ ]:
# ── Cell 5: Quick sanity-check (one batch through model) ──────────────────────
import sys
import yaml
import torch
from pathlib import Path

# Make sure project src is importable
if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))

config = yaml.safe_load(Path('configs/config.yaml').read_text())

# Quick check config
print('Config loaded:')
print(f"  batch_size  : {config['training']['batch_size']}")
print(f"  num_workers : {config['training']['num_workers']}")
print(f"  phase1_epochs: {config['training']['phase1_epochs']}")
print(f"  phase2_epochs: {config['training']['phase2_epochs']}")
print(f"  backbone    : {config['model']['backbone']}")

from src.models.dual_branch import DualBranchModel
from src.preprocessing.srm_filters import SRMFilterLayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

# Build model
model = DualBranchModel(
    backbone=config['model']['backbone'],
    pretrained_rgb=True,
    pretrained_noise=False,
    feature_dim=config['model']['feature_dim'],
    hidden_dim=config['model']['hidden_dim'],
    dropout=config['model']['dropout'],
    freeze_bn=config['training']['freeze_bn'],
).to(device)

# Test one dummy forward pass
rgb_dummy   = torch.randn(2, 3, 224, 224).to(device)
noise_dummy = torch.randn(2, 33, 224, 224).to(device)
out = model(rgb_dummy, noise_dummy)
print(f'Model forward pass OK. Output shape: {out.shape}')  # should be [2, 1]

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── Cell 6: Test dataloader (loads one real batch) ────────────────────────────
import yaml
from pathlib import Path
from src.preprocessing.dataset_builder import create_dataloaders
from src.preprocessing.srm_filters import SRMFilterLayer

config = yaml.safe_load(Path('configs/config.yaml').read_text())

# Use 0 workers for this quick test to get clean error messages
srm_layer = SRMFilterLayer().to(device)
train_loader, val_loader, test_loader = create_dataloaders(
    config,
    batch_size=4,
    num_workers=0,
    srm_layer=srm_layer,
)

print(f'Train: {len(train_loader.dataset):,} samples  ({len(train_loader)} batches @ bs=4)')
print(f'Val  : {len(val_loader.dataset):,} samples')
print(f'Test : {len(test_loader.dataset):,} samples')

batch = next(iter(train_loader))
print(f'\nFirst batch shapes:')
print(f"  rgb   : {batch['rgb'].shape}")
print(f"  noise : {batch['noise'].shape}")
print(f"  label : {batch['label'].shape}")
print(f"  label values: {batch['label'].flatten().tolist()}")
print('\nDataloader OK ✓')

In [ ]:
# ── Cell 7: START TRAINING ────────────────────────────────────────────────────
# This runs the full training pipeline:
#   Phase 1 (10 epochs): warms up the noise branch
#   Phase 2 (40 epochs): full model fine-tuning
# Checkpoints are saved to models/checkpoints/ every epoch.
# Best model is saved to models/checkpoints/best_model.pth

!python train.py

In [ ]:
# ── Cell 8 (Optional): Resume training from a checkpoint ─────────────────────
# If training was interrupted, resume from the last saved checkpoint:

# !python train.py --resume models/checkpoints/best_model.pth

In [ ]:
# ── Cell 9 (Optional): Plot training curves ───────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path('logs/training_log.csv')
if not log_path.exists():
    print('No training log found yet. Run Cell 7 first.')
else:
    df = pd.read_csv(log_path)
    print(df.tail(10).to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('DeepSight Training Curves', fontsize=14, fontweight='bold')

    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(df['epoch'], df['val_auc'], color='green', label='Val AUC')
    axes[1].set_title('Validation AUC-ROC'); axes[1].legend(); axes[1].grid(True)

    axes[2].plot(df['epoch'], df['val_f1'],  color='orange', label='Val F1')
    axes[2].plot(df['epoch'], df['val_acc'], color='purple', label='Val Acc')
    axes[2].set_title('Val F1 & Accuracy'); axes[2].legend(); axes[2].grid(True)

    plt.tight_layout()
    plt.savefig('logs/training_curves.png', dpi=150)
    plt.show()
    print('Saved → logs/training_curves.png')

In [ ]:
# ── Cell 10 (Optional): Copy best model back to Drive ─────────────────────────
import shutil, os

BEST_CKPT = 'models/checkpoints/best_model.pth'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/deepsight_checkpoints'

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
if os.path.exists(BEST_CKPT):
    shutil.copy(BEST_CKPT, DRIVE_SAVE_DIR)
    print(f'Best model saved to Drive: {DRIVE_SAVE_DIR}/best_model.pth')
else:
    print(f'No checkpoint found at {BEST_CKPT}')